# 03 — Personnalisation du LLM (fine-tuning LoRA)
Étape 2 de l'énoncé. Méthode choisie : **SFT avec adaptateur LoRA** sur `Qwen/Qwen3-8B` (open weights), entrée = prompts du dossier `prompts/`, sortie = nom exact de la file ; perte uniquement sur la réponse.

Protocole (sans regarder le test) :
1. **Version non personnalisée sur la validation** : même modèle, sans adaptateur, avec le prompt complet (définitions) puis le prompt compact.
2. **Choix des hyperparamètres** sur la validation (entraînement sur `train`, mesure à chaque époque).
3. **Modèle final** : réentraînement sur `train + val` avec la configuration retenue, 3 graines. Le test n'est évalué qu'au notebook 04.

In [1]:
import sys, os, json, time, warnings
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 90)
from classif import config, data, prompts, evaluation
config.RESULTS_DIR.mkdir(exist_ok=True)
import torch
from sklearn.metrics import f1_score
from classif import hf_model, finetune
train, val, test = data.load_splits()
print(len(train), len(val), len(test))

384 96 120


## 1. Référence sur la validation : le modèle de base, sans personnalisation

In [2]:
tok = hf_model.load_tokenizer()
base = hf_model.load_model()
for compact in (False, True):
    pred0 = hf_model.predict(hf_model.score(base, tok, val, compact=compact))
    f = f1_score(val.queue, pred0, average="weighted")


    print(f"Qwen3-8B sans personnalisation, prompt {'compact' if compact else 'complet'} — validation : F1 pondéré {f:.3f}, exactitude {np.mean(np.array(pred0) == val.queue.values):.3f}")
    if not compact: f1_base_val = f
del base; torch.cuda.empty_cache()

Qwen3-8B sans personnalisation, prompt complet — validation : F1 pondéré 0.378, exactitude 0.427


Qwen3-8B sans personnalisation, prompt compact — validation : F1 pondéré 0.261, exactitude 0.271


## 2. Recherche d'hyperparamètres sur la validation
**Première série (archivée)** : prompt complet, 3 époques, lr 1e-4 puis 2e-4. La validation montait encore (F1 0,626 puis 0,756 ; perte d'entraînement 0,19 puis 0,12 : le modèle n'avait pas fini d'apprendre) mais chaque époque coûtait ≈ 9 min. J'ai interrompu avant l'entraînement final et réduit le coût (prompt compact, ≈ 2× plus rapide) pour entraîner plus longtemps.

**Série retenue** : prompt compact, rang LoRA 16, un entraînement sur `train` par configuration, F1 pondéré mesuré sur `val` à chaque époque.

In [3]:
pd.read_csv(config.RESULTS_DIR / "recherche_hyperparametres_prompt_complet.csv").round(3)

,lr,epochs,epoch,train_loss,secondes,val_f1_pondere,val_exactitude
0,0.0,3,1,0.751,534,0.485,0.510
1,0.0,3,2,0.305,1106,0.577,0.604
2,0.0,3,3,0.186,1677,0.626,0.646
3,0.0,3,1,0.655,539,0.502,0.542
4,0.0,3,2,0.277,1109,0.591,0.625
5,0.0,3,3,0.116,1680,0.756,0.760


In [4]:
GRILLE = [{"lr": 2e-4, "epochs": 6}, {"lr": 4e-4, "epochs": 4}]
essais = []
for cfg in GRILLE:
    print("===", cfg)
    hist = finetune.train(train, val, out_dir=None, rank=16, seed=0, **cfg)
    for h in hist:
        essais.append({**cfg, **h})
    pd.DataFrame(essais).to_csv(config.RESULTS_DIR / "recherche_hyperparametres.csv", index=False)
essais = pd.DataFrame(essais)
essais

=== {'lr': 0.0002, 'epochs': 6}


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


{"epoch": 1, "train_loss": 0.7406104964514574, "secondes": 244, "val_f1_pondere": 0.5579143018641566, "val_exactitude": 0.59375}


{"epoch": 2, "train_loss": 0.3197074863225377, "secondes": 508, "val_f1_pondere": 0.6513981871198494, "val_exactitude": 0.6770833333333334}


{"epoch": 3, "train_loss": 0.12751286045022425, "secondes": 771, "val_f1_pondere": 0.7798611111111112, "val_exactitude": 0.78125}


...

{"epoch": 1, "train_loss": 0.6212252774275839, "secondes": 244, "val_f1_pondere": 0.428044243853771, "val_exactitude": 0.4479166666666667}


{"epoch": 2, "train_loss": 0.27327188198978547, "secondes": 506, "val_f1_pondere": 0.7145836419143543, "val_exactitude": 0.7291666666666666}


{"epoch": 3, "train_loss": 0.08706217750528822, "secondes": 769, "val_f1_pondere": 0.8207232019598724, "val_exactitude": 0.8333333333333334}


{"epoch": 4, "train_loss": 0.019969239690112772, "secondes": 1030, "val_f1_pondere": 0.8930077597840755, "val_exactitude": 0.8958333333333334}


,lr,epochs,epoch,train_loss,secondes,val_f1_pondere,val_exactitude
0,0.0002,6,1,0.740610,244,0.557914,0.593750
1,0.0002,6,2,0.319707,508,0.651398,0.677083
2,0.0002,6,3,0.127513,771,0.779861,0.781250
3,0.0002,6,4,0.051646,1032,0.775579,0.781250
4,0.0002,6,5,0.002255,1295,0.829279,0.833333
5,0.0002,6,6,0.000271,1558,0.829279,0.833333
6,0.0004,4,1,0.621225,244,0.428044,0.447917
7,0.0004,4,2,0.273272,506,0.714584,0.729167
8,0.0004,4,3,0.087062,769,0.820723,0.833333
9,0.0004,4,4,0.019969,1030,0.893008,0.895833


In [5]:
final_ep = essais.sort_values(["epoch"]).groupby(["lr", "epochs"]).tail(1)   # F1 en fin d'entraînement de chaque configuration
display(final_ep[["lr", "epochs", "train_loss", "val_f1_pondere", "val_exactitude", "secondes"]].round(3))
meilleur = final_ep.sort_values(["val_f1_pondere", "epochs"], ascending=[False, True]).iloc[0]
CFG = {"lr": float(meilleur.lr), "epochs": int(meilleur.epochs)}
print("Configuration retenue (meilleur F1 pondéré en validation) :", CFG, f"| val F1 {meilleur.val_f1_pondere:.3f} contre {f1_base_val:.3f} sans personnalisation")
json.dump({**CFG, "rank": 16, "alpha": 32, "dropout": 0.05, "batch_size": 4, "grad_accum": 2, "base_model": config.BASE_MODEL,
           "val_f1_base": f1_base_val, "val_f1_retenu": float(meilleur.val_f1_pondere)}, open(config.RESULTS_DIR / "config_retenue.json", "w"), indent=1)

,lr,epochs,train_loss,val_f1_pondere,val_exactitude,secondes
9,0.0,4,0.02,0.893,0.896,1030
5,0.0,6,0.00,0.829,0.833,1558


Configuration retenue (meilleur F1 pondéré en validation) : {'lr': 0.0004, 'epochs': 4} | val F1 0.893 contre 0.378 sans personnalisation


## 3. Modèle final : réentraînement sur train + validation, 3 graines
La graine 0 est désignée **à l'avance** comme modèle de référence ; les 3 résultats sont rapportés au notebook 04 pour montrer la variabilité d'un entraînement à l'autre (aucun choix de graine sur le test).

In [6]:
trval = pd.concat([train, val])
logs = []
for seed in (0, 1, 2):
    print("=== graine", seed)
    hist = finetune.train(trval, None, out_dir=config.MODELS_DIR / f"lora_seed{seed}", rank=16, seed=seed, **CFG)
    logs += [{"seed": seed, **h} for h in hist]
    pd.DataFrame(logs).to_csv(config.RESULTS_DIR / "entrainement_final.csv", index=False)
pd.DataFrame(logs)

=== graine 0


{"epoch": 1, "train_loss": 0.5949627765764792, "secondes": 303}


{"epoch": 2, "train_loss": 0.2570483622471026, "secondes": 607}


{"epoch": 3, "train_loss": 0.08588818204064713, "secondes": 912}


{"epoch": 4, "train_loss": 0.010669070095688463, "secondes": 1215}


...

{"epoch": 1, "train_loss": 0.6761948933824897, "secondes": 300}


{"epoch": 2, "train_loss": 0.3327880585566163, "secondes": 606}


{"epoch": 3, "train_loss": 0.11246256437055611, "secondes": 910}


{"epoch": 4, "train_loss": 0.01096514603773926, "secondes": 1213}


,seed,epoch,train_loss,secondes
0,0,1,0.594963,303
1,0,2,0.257048,607
2,0,3,0.085888,912
3,0,4,0.010669,1215
4,1,1,0.602261,303
5,1,2,0.268429,606
6,1,3,0.091455,909
7,1,4,0.018621,1211
8,2,1,0.676195,300
9,2,2,0.332788,606


In [7]:
print([p.name for p in sorted(config.MODELS_DIR.glob('lora_seed*'))])
for p in sorted(config.MODELS_DIR.glob('lora_seed*')):
    print(p.name, round(sum(f.stat().st_size for f in p.iterdir()) / 2**20, 1), "Mo")

['lora_seed0', 'lora_seed1', 'lora_seed2']
lora_seed0 166.6 Mo
lora_seed1 166.6 Mo
lora_seed2 166.6 Mo
